In [1]:
import matplotlib
import numpy as np
matplotlib.use("Agg")
from IPython.display import display, Video
from stable_baselines3 import DDPG
from wheelchair_env import WheelchairNavEnv
import imageio.v2 as imageio
import random
import numpy as np
import torch


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Load model ────────────────────────────────────────────────────────────────
model = DDPG.load('finalzip/ddpg_wheelchair_OUv2.zip')

env = WheelchairNavEnv(
    action_type='continuous',
    n_people=3,
    n_obstacles=4,
    render_mode='rgb_array',
    seed=SEED
)

N_EVAL = 1000
eval_rewards = []
eval_success = []
eval_collisions = []
eval_timeouts = []


for ep in range(N_EVAL):
    obs, _ = env.reset(seed=SEED + ep)
    total_reward = 0.0
    done = False

    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        done = terminated or truncated

    # check outcome only after episode ends
    dist = info.get("dist_to_goal", float("inf"))

    success   = int(terminated and dist <= 0.4)
    collision = int(terminated and dist > 0.4)
    timeout   = int(truncated)

    eval_rewards.append(total_reward)
    eval_success.append(success)
    eval_collisions.append(collision)
    eval_timeouts.append(timeout)

    print(f"Ep {ep+1:>3} | reward: {total_reward:>8.2f} | {'GOAL' if success else 'COLLISION' if collision else 'TIMEOUT'}")
print()
print(f"  Total reward   : {np.sum(eval_rewards):.2f}")
print(f"  Average reward : {np.mean(eval_rewards):.2f}")
print(f"  Success rate : {np.mean(eval_success)*100:.2f} %")
print(f"  Collision rate: {np.mean(eval_collisions)*100:.2f} %")
#print(f"  Timeout rate  : {np.mean(eval_timeouts)*100:.2f} %")

Ep   1 | reward:   168.66 | GOAL
Ep   2 | reward:   148.24 | GOAL
Ep   3 | reward:   125.82 | GOAL
Ep   4 | reward:    43.97 | COLLISION
Ep   5 | reward:   -26.90 | GOAL
Ep   6 | reward:   147.28 | GOAL
Ep   7 | reward:  -187.08 | GOAL
Ep   8 | reward:   126.67 | GOAL
Ep   9 | reward:   157.08 | GOAL
Ep  10 | reward:   197.94 | GOAL
Ep  11 | reward:    47.60 | GOAL
Ep  12 | reward:   191.66 | GOAL
Ep  13 | reward:   -84.61 | GOAL
Ep  14 | reward:   122.92 | COLLISION
Ep  15 | reward:   235.75 | GOAL
Ep  16 | reward:   108.92 | COLLISION
Ep  17 | reward:   202.06 | GOAL
Ep  18 | reward:   187.59 | GOAL
Ep  19 | reward:    42.35 | COLLISION
Ep  20 | reward:    71.40 | GOAL
Ep  21 | reward:    46.37 | GOAL
Ep  22 | reward:   202.14 | GOAL
Ep  23 | reward:   153.02 | COLLISION
Ep  24 | reward:   247.32 | COLLISION
Ep  25 | reward:   296.85 | COLLISION
Ep  26 | reward:    -1.77 | COLLISION
Ep  27 | reward:   108.60 | GOAL
Ep  28 | reward:   242.31 | GOAL
Ep  29 | reward:    93.94 | GOAL
Ep 